# Explainable-Hybrid-ASAG - full neural + hybrid run (Kaggle / GPU)

Kaggle-native version (no Google Drive, no `google.colab`). Runs the deferred torch slice end-to-end and
writes every artifact the paper's `\TBD{}` slots need: Phase 2G (DeBERTa), Phase 4 hybrid OOF, downstream
(ablations -D / three-way / train2d / xai), and the LLM baseline.

### Before you run (do these in the right panel / Settings)
1. **Add your project as a Dataset:** right panel -> **Add Input** -> Datasets -> upload/select the dataset that
   contains this project (the folder with `pyproject.toml`, `src/`, `configs/`, and `data/processed/<ds>/{encoder,features}.parquet`).
   It will mount read-only under `/kaggle/input/...`.
2. **Accelerator:** Settings -> **GPU P100** (or **GPU T4 x2**).
3. **Internet:** Settings -> **Internet: On** (needed for pip + model downloads; requires phone-verified account).
4. **Persistence:** Settings -> **Persistence: Files only** -> so `/kaggle/working` survives across sessions and the
   run RESUMES after any interruption instead of restarting.

Then **Run All** (or **Save Version -> Save & Run All (Commit)** to run headless in the background).
Every stage is **resumable** - a finished dataset is never re-done. Results land in `/kaggle/working`.

In [ ]:
# 1) Deps. Kaggle ships torch + transformers (GPU build) - we do NOT touch them. Add only what the
#    project needs, install the spaCy model, and drop torchao (a stale copy breaks fresh peft's LoRA).
%pip -q install peft sentence-transformers sentencepiece protobuf accelerate loguru "lightgbm==4.5.0" "optuna==4.1.0"
!python -m spacy download en_core_web_sm
!pip -q uninstall -y torchao
print("deps ready - torch/transformers left as Kaggle shipped them")

In [ ]:
# 2) Locate the project you added as a Dataset (under /kaggle/input) and copy it into /kaggle/working
#    (writable + persisted). Robust whether Kaggle auto-extracted the zip or you added an extracted folder.
import os, sys, glob, shutil, subprocess
srcs = glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
if not srcs:                                   # a raw .zip that Kaggle did not extract -> unzip it
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'No project under /kaggle/input. Right panel -> Add Input -> your project Dataset.'
    subprocess.run(['unzip', '-q', '-o', zips[0], '-d', '/kaggle/working/_unz'], check=True)
    srcs = glob.glob('/kaggle/working/_unz/**/pyproject.toml', recursive=True)
SRC = os.path.dirname(next(p for p in srcs
                           if os.path.exists(os.path.join(os.path.dirname(p), 'configs', 'data.yaml'))))
REPO = '/kaggle/working/asag'
if not os.path.exists(os.path.join(REPO, 'pyproject.toml')):
    shutil.copytree(SRC, REPO, dirs_exist_ok=True)   # writable working copy (config + outputs write here)
print('SRC  =', SRC)
print('REPO =', REPO)

In [ ]:
# 3) Env + config. Put src on the path, point the config at the working repo, set the run knobs.
#    SEEDS: keep [42,1,2] for publication-grade mean+-std; set [42] for the shortest guaranteed pass.
SEEDS = [42, 1, 2]     # <-- change to [42] if you want the fastest single-session run
os.environ['ASAG_PROJECT_ROOT'] = REPO
os.environ['PYTHONPATH'] = os.path.join(REPO, 'src')
os.environ['PYTHONUTF8'] = '1'; os.environ['PYTHONIOENCODING'] = 'utf-8'; os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, 'src'))
import yaml, torch
p = os.path.join(REPO, 'configs', 'data.yaml'); d = yaml.safe_load(open(p, encoding='utf-8'))
d['neural'] = {**d.get('neural', {}), 'enabled': True, 'backbone': 'microsoft/deberta-v3-base', 'device': 'cuda',
               'epochs': 4, 'batch_size': 16, 'lr': 2.0e-5, 'warmup_ratio': 0.1, 'dropout': 0.1,
               'ordinal_head': 'corn', 'pooling': 'cls', 'seeds': SEEDS, 'num_workers': 0,
               'select_on_dev': True, 'save_predictions': True, 'lora_enabled': True, 'lora_r': 8,
               'lora_alpha': 16, 'lora_dropout': 0.1, 'max_len': 128, 'max_len_overrides': {'saf': 256}}
yaml.safe_dump(d, open(p, 'w', encoding='utf-8'), sort_keys=False, allow_unicode=True)
proc = os.path.join(REPO, 'data', 'processed'); need = ('encoder.parquet', 'features.parquet')
present = sorted(x for x in os.listdir(proc) if os.path.isdir(os.path.join(proc, x))
                 and all(os.path.exists(os.path.join(proc, x, f)) for f in need)) if os.path.isdir(proc) else []
assert present, 'No processed parquets. Your dataset must include data/processed/<ds>/{encoder,features}.parquet.'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE',
      '| seeds:', SEEDS, '| num_workers: 0 | datasets:', present)

In [ ]:
# 4) Fail-fast + SMOKE (~3 min, one-time): actually fine-tune DeBERTa+LoRA on a tiny slice per task
#    family and load SBERT + spaCy - so any incompatibility surfaces now, not hours into the real run.
import transformers, peft, lightgbm, optuna, sentence_transformers, numpy as np, pandas as pd
from transformers import AutoTokenizer
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'No GPU. Settings -> Accelerator -> GPU (P100 or T4 x2), then re-run.'
import asag
from asag.config import load_data_config
load_data_config.cache_clear(); cfg = load_data_config()
assert cfg.neural.device == 'cuda' and cfg.neural.lora_enabled
_tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
from asag.models.tasks import get_spec
from asag.neural.evaluate_neural import load_text_df
from asag.neural.trainer import fit_predict
def _tiny(df, pg=8, n=28):
    key = (df['label'].astype(str) if ('label' in df.columns and df['label'].notna().any())
           else pd.to_numeric(df['score'], errors='coerce').round())
    s = pd.concat([g.head(pg) for _, g in df.groupby(key)]).head(n).sample(frac=1.0, random_state=42).reset_index(drop=True)
    c = max(4, int(len(s) * 0.7)); return s.iloc[:c].reset_index(drop=True), s.iloc[c:].reset_index(drop=True)
sm = cfg.neural.model_copy(update={'epochs': 1, 'batch_size': 8, 'num_workers': 0, 'select_on_dev': False})
for ds in ('mohler', 'mindreading', 'powergrading'):
    if ds not in present: continue
    tr, te = _tiny(load_text_df(ds, cfg))
    r = fit_predict(tr, te, get_spec(ds), sm, seed=42, max_len=64, tokenizer=_tok)
    assert np.isfinite(np.asarray(r['y_cont'])).all() and len(r['y_pred']) == len(te)
    print('  smoke ok:', ds, '[' + get_spec(ds).task_type + ']')
from asag.features.semantic import SbertEncoder
from asag.features.text_utils import load_feature_nlp
SbertEncoder(cfg.features.sbert_model, batch_size=8, normalize=True).embed(['a short answer', 'another'])
load_feature_nlp(cfg.features.ner.spacy_model)
print('SMOKE OK - neural + SBERT + spaCy all work on this Kaggle runtime')

In [ ]:
# 5) Driver. Small -> large so early progress is visible; every stage below skips finished work.
DATASETS = [x for x in ['mohler', 'powergrading', 'mindreading', 'saf', 'asap_sas', 'semeval'] if x in present]
def run(*a):
    print('>> python -m', *a, flush=True)
    return subprocess.run([sys.executable, '-m', *a], cwd=REPO, env=dict(os.environ)).returncode
print('will process:', DATASETS)

In [ ]:
# 6) Phase 2G - DeBERTa vs GBM. RESUMABLE: each finished dataset is cached under reports/phase2g/_partial
#    in /kaggle/working. With Persistence=Files only, a session time-out never costs a completed dataset -
#    just re-run this cell (or re-commit) and it resumes from where it stopped.
import json
from asag.config import ensure_dirs
from asag.neural import NEURAL_SCHEMA_VERSION
from asag.neural.run import _write_figure
ensure_dirs(cfg)
p2g = os.path.join(REPO, 'reports', 'phase2g'); part = os.path.join(p2g, '_partial'); os.makedirs(part, exist_ok=True)
for ds in DATASETS:
    cache = os.path.join(part, ds + '.json')
    if os.path.exists(cache):
        print('skip (done):', ds); continue
    if run('asag.neural.run', ds) == 0 and os.path.exists(os.path.join(p2g, 'results.json')):
        b = json.load(open(os.path.join(p2g, 'results.json'))).get('datasets', {}).get(ds)
        if b:
            json.dump(b, open(cache, 'w'), default=str); print('  cached', ds)
blocks = {ds: json.load(open(os.path.join(part, ds + '.json')))
          for ds in DATASETS if os.path.exists(os.path.join(part, ds + '.json'))}
if blocks:
    json.dump({'schema_version': NEURAL_SCHEMA_VERSION, 'backbone': cfg.neural.backbone,
               'seeds': list(cfg.neural.seeds), 'datasets': blocks},
              open(os.path.join(p2g, 'results.json'), 'w'), indent=2, default=str)
    _write_figure(cfg, blocks); print('Phase 2G done for:', list(blocks))
else:
    print('Phase 2G produced no blocks - check the logs above')

In [ ]:
# 7) Phase 4 hybrid OOF -> data/processed/<ds>/neural_oof.parquet (resumable per dataset).
for ds in DATASETS:
    oof = os.path.join(REPO, 'data', 'processed', ds, 'neural_oof.parquet')
    if os.path.exists(oof):
        print('skip (done):', ds); continue
    run('asag.neural.extract_features', ds)

In [ ]:
# 8) Verify OOF coverage (~100% each).
import pandas as pd, numpy as np
for ds in DATASETS:
    oof = os.path.join(REPO, 'data', 'processed', ds, 'neural_oof.parquet')
    if os.path.exists(oof):
        d = pd.read_parquet(oof)
        print(ds.ljust(14), 'rows=%6d' % len(d), 'coverage=%.1f%%' % (100 * np.isfinite(d['neural_score']).mean()))
    else:
        print(ds.ljust(14), 'MISSING - re-run cell 7')

In [ ]:
# 9) Downstream GBM-with-neural (fast). load_bundle auto-concatenates neural_* -> the hybrid everywhere.
#    Each runs as a subprocess; a non-zero exit is reported but non-fatal (neural artifacts already saved).
summary = {}
for mod in ('asag.models.ablations', 'asag.models.neural_compare', 'asag.models.train2d', 'asag.xai.run'):
    summary[mod] = run(mod)
print('downstream exit codes:', summary)
print('OK' if not [m for m, c in summary.items() if c] else 'WARNING: nonzero exits - scroll up')

In [ ]:
LLM_BASELINE_B64 = (
    'IiIiWmVyby1zaG90IExMTSBncmFkaW5nIGJhc2VsaW5lIHVuZGVyIHRoZSBwYXBlcidzIG93biBtZXRyaWNzL3Byb3RvY29sLgoKR3JhZGVzIGVhY2ggZGF0YXNldCdzICoqaGVhZGxpbmUgc3BsaXQqKiB3aXRoIGFuIGluc3RydWN0aW9uLXR1bmVkIExMTSAodGVtcGVyYXR1cmUKMCwgcHJvbXB0cyBsb2dnZWQpLCB0aGVuIHNjb3JlcyBpdCB3aXRoIHRoZSAqc2FtZSogbWV0cmljIHRoZSBHQk0vRGVCRVJUYSBhcm1zIHVzZQooYGBhc2FnLm1vZGVscy5tZXRyaWNzYGAgdmlhIGBgYXNhZy5tb2RlbHMudGFza3MuUkVHSVNUUllgYCkuIFRoaXMgaXMgdGhlIExMTSByb3cKcmV2aWV3ZXJzIGV4cGVjdCBpbiAyMDI2OyB0aGUgcHJvamVjdCdzIG93biByZXZpZXcgc2ltdWxhdGlvbiBmbGFncyBpdHMgYWJzZW5jZSBhcyBhCm5lYXItcmVqZWN0IGF0IFEyIChSZXZpZXdlciBDKS4KCkRlc2lnbiBjaG9pY2VzIChhbGwgZGVsaWJlcmF0ZSwgYWxsIGRvY3VtZW50ZWQgc28gYSByZXZpZXdlciBjYW4gY2hlY2spOgoKKiAqKlplcm8tc2hvdCwgbm8gdHJhaW5pbmcsIG5vIGZvbGRzLioqIGBgb2ZmaWNpYWxfc3BsaXRgYCBkYXRhc2V0cyBhcmUgZ3JhZGVkIG9uIHRoZQogIGxhc3QgLyBoYXJkZXN0IHRlc3Qgc3BsaXQgKFNlbUV2YWwgYGB0ZXN0X3VkYGAsIFNBRiBgYHRlc3RfdXFgYCwgQVNBUC1TQVMKICBgYHRlc3RfdWFgYCk7IGBga2ZvbGRgYCBkYXRhc2V0cyAoTW9obGVyLCBQb3dlcmdyYWRpbmcsIE1JTkQtQ0EpIGFyZSBncmFkZWQgb25jZSBvbgogIGFsbCBDViByb3dzIOKAlCB0aGVyZSBpcyBub3RoaW5nIHRvIGZpdCwgc28gdGhlIGZvbGQgc3RydWN0dXJlIGlzIGlycmVsZXZhbnQuCiogKipUYXNrLWF3YXJlIHByb21wdCArIG1ldHJpYy4qKiBjbGFzc2lmaWNhdGlvbiDihpIgcGljayBvbmUgbGFiZWwgKG1hY3JvLUYxIG9uIHNoYXJlZAogIGNvZGVzKTsgb3JkaW5hbCDihpIgYW4gaW50ZWdlciBpbiB0aGUgb2JzZXJ2ZWQgcmFuZ2UgKFFXSyk7IHJlZ3Jlc3Npb24g4oaSIGEgbnVtYmVyIGluCiAgcmFuZ2UgKFBlYXJzb24pLiBQZXItcHJvbXB0IGRhdGFzZXRzIChBU0FQLVNBUykgYXZlcmFnZSB0aGUgbWV0cmljIGFjcm9zcyBwcm9tcHRzLAogIGV4YWN0bHkgbGlrZSB0aGUgR0JNIGhlYWRsaW5lLgoqICoqQ29zdCBjYXAuKiogYGBMTE1fTUFYX1JPV1NgYCAoZGVmYXVsdCA0MDApIHN1YnNhbXBsZXMgbGFyZ2UgdGVzdCBzcGxpdHMgd2l0aCBhCiAgZml4ZWQgc2VlZDsgYGBuX2V2YWxgYCBpcyByZXBvcnRlZCBzbyB0aGUgZXN0aW1hdGUncyBiYXNpcyBpcyBleHBsaWNpdC4KClN3YXAgYGBMTE1fTU9ERUxgYCBmb3IgYSBzdHJvbmdlciBvcGVuIG1vZGVsICg3Qi84Qikgb3Igd2lyZSBhbiBBUEkgaW5zaWRlCmBgZ2VuZXJhdGUoKWBgIGZvciB0aGUgaGVhZGxpbmUgcm93OyB0aGUgbWV0cmljIHBsdW1iaW5nIGlzIHVuY2hhbmdlZC4KCiAgICBweXRob24gZXhwZXJpbWVudHMvbGxtX3plcm9zaG90X2Jhc2VsaW5lLnB5IFtkYXRhc2V0IC4uLl0KICAgIExMTV9NT0RFTD1Rd2VuL1F3ZW4yLjUtM0ItSW5zdHJ1Y3QgTExNX01BWF9ST1dTPTQwMCBweXRob24gZXhwZXJpbWVudHMvbGxtX3plcm9zaG90X2Jhc2VsaW5lLnB5CgpPdXRwdXRzOiByZXBvcnRzL3BoYXNlX2xsbS97bGxtX2Jhc2VsaW5lLmpzb24sIGV4YW1wbGVzLmpzb259LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc3lzCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKZnJvbSBhc2FnLmNvbmZpZyBpbXBvcnQgbG9hZF9kYXRhX2NvbmZpZwpmcm9tIGFzYWcubW9kZWxzLm1ldHJpY3MgaW1wb3J0IGNvbXB1dGVfbWV0cmljcwpmcm9tIGFzYWcubW9kZWxzLnRhc2tzIGltcG9ydCBSRUdJU1RSWSwgZ2V0X3NwZWMKCk1PREVMID0gb3MuZW52aXJvbi5nZXQoIkxMTV9NT0RFTCIsICJRd2VuL1F3ZW4yLjUtM0ItSW5zdHJ1Y3QiKQpNQVhfUk9XUyA9IGludChvcy5lbnZpcm9uLmdldCgiTExNX01BWF9ST1dTIiwgIjQwMCIpKSAgICAgICAgICAjIGNhcCBldmFsIHJvd3MvZGF0YXNldCAoMCA9IGFsbCkKTUFYX1BFUl9QUk9NUFQgPSBpbnQob3MuZW52aXJvbi5nZXQoIkxMTV9NQVhfUEVSX1BST01QVCIsICIxNTAiKSkKQkFUQ0ggPSBpbnQob3MuZW52aXJvbi5nZXQoIkxMTV9CQVRDSCIsICIxNiIpKQpTRUVEID0gNDIKCgpkZWYgX2V2YWxfcm93cyhkZjogcGQuRGF0YUZyYW1lLCBzcGVjKSAtPiBwZC5EYXRhRnJhbWU6CiAgICAiIiJIZWFkbGluZS1zcGxpdCByb3dzIGZvciBvZmZpY2lhbF9zcGxpdDsgYWxsIENWIHJvd3MgZm9yIGtmb2xkLiBDb3N0LWNhcHBlZC4iIiIKICAgIGlmIHNwZWMucHJvdG9jb2wgPT0gImtmb2xkIjoKICAgICAgICBtYXNrID0gcGQudG9fbnVtZXJpYyhkZlsiZm9sZCJdLCBlcnJvcnM9ImNvZXJjZSIpLmZpbGxuYSgtMSkgPj0gMAogICAgZWxzZToKICAgICAgICBtYXNrID0gZGZbInNwbGl0Il0uYXN0eXBlKHN0cikgPT0gc3BlYy50ZXN0X3NwbGl0c1stMV0KICAgIGQgPSBkZlttYXNrXS5yZXNldF9pbmRleChkcm9wPVRydWUpCiAgICBpZiBzcGVjLnBlcl9wcm9tcHQgYW5kIE1BWF9QRVJfUFJPTVBUOgogICAgICAgIGQgPSAoZC5ncm91cGJ5KCJxdWVzdGlvbl9pZCIsIGdyb3VwX2tleXM9RmFsc2UpCiAgICAgICAgICAgICAgIC5hcHBseShsYW1iZGEgZzogZy5zYW1wbGUobj1taW4obGVuKGcpLCBNQVhfUEVSX1BST01QVCksIHJhbmRvbV9zdGF0ZT1TRUVEKSkKICAgICAgICAgICAgICAgLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkpCiAgICBlbGlmIE1BWF9ST1dTIGFuZCBsZW4oZCkgPiBNQVhfUk9XUzoKICAgICAgICBkID0gZC5zYW1wbGUobj1NQVhfUk9XUywgcmFuZG9tX3N0YXRlPVNFRUQpLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIHJldHVybiBkCgoKZGVmIF9sYWJlbF9zcGFjZShkZjogcGQuRGF0YUZyYW1lLCBzcGVjKSAtPiBkaWN0OgogICAgIiIiQWxsb3dlZCBsYWJlbHMgKGNsYXNzaWZpY2F0aW9uKSBvciBudW1lcmljIHJhbmdlIChvcmRpbmFsL3JlZ3Jlc3Npb24pLiIiIgogICAgaWYgc3BlYy50YXNrX3R5cGUgPT0gImNsYXNzaWZpY2F0aW9uIjoKICAgICAgICBsYWJzID0gc29ydGVkKHMgZm9yIHMgaW4gZGZbImxhYmVsIl0uYXN0eXBlKHN0cikudW5pcXVlKCkgaWYgcyBhbmQgcy5sb3dlcigpICE9ICJuYW4iKQogICAgICAgIHJldHVybiB7ImxhYmVscyI6IGxhYnMsICJ2b2NhYiI6IHtsYWI6IGkgZm9yIGksIGxhYiBpbiBlbnVtZXJhdGUobGFicyl9fQogICAgcyA9IHBkLnRvX251bWVyaWMoZGZbInNjb3JlIl0sIGVycm9ycz0iY29lcmNlIikuZHJvcG5hKCkKICAgIHJldHVybiB7ImxvIjogZmxvYXQocy5taW4oKSksICJoaSI6IGZsb2F0KHMubWF4KCkpfQoKCmRlZiBfcHJvbXB0KHJvdzogcGQuU2VyaWVzLCBzcGVjLCBzcGFjZTogZGljdCkgLT4gc3RyOgogICAgcSA9IHN0cihyb3cuZ2V0KCJxdWVzdGlvbl9lbmMiKSBvciAiIikuc3RyaXAoKQogICAgcmVmID0gc3RyKHJvdy5nZXQoInJlZmVyZW5jZV9hbnN3ZXJfZW5jIikgb3IgIiIpLnN0cmlwKCkKICAgIGFucyA9IHN0cihyb3cuZ2V0KCJzdHVkZW50X2Fuc3dlcl9lbmMiKSBvciAiIikuc3RyaXAoKQogICAgY3R4ID0gIiIKICAgIGlmIHEgYW5kIHEubG93ZXIoKSAhPSAibmFuIjoKICAgICAgICBjdHggKz0gZiJRdWVzdGlvbjoge3F9XG4iCiAgICBpZiByZWYgYW5kIHJlZi5sb3dlcigpICE9ICJuYW4iOgogICAgICAgIGN0eCArPSBmIlJlZmVyZW5jZSBhbnN3ZXI6IHtyZWZ9XG4iCiAgICBpZiBzcGVjLnRhc2tfdHlwZSA9PSAiY2xhc3NpZmljYXRpb24iOgogICAgICAgIHRhc2sgPSAoZiJDbGFzc2lmeSB0aGUgc3R1ZGVudCBhbnN3ZXIgaW50byBleGFjdGx5IG9uZSBvZiB0aGVzZSBsYWJlbHM6ICIKICAgICAgICAgICAgICAgIGYieycsICcuam9pbihzcGFjZVsnbGFiZWxzJ10pfS5cblJlcGx5IHdpdGggT05MWSB0aGUgbGFiZWwsIG5vdGhpbmcgZWxzZS4iKQogICAgZWxpZiBzcGVjLnRhc2tfdHlwZSA9PSAib3JkaW5hbCI6CiAgICAgICAgdGFzayA9IChmIkdyYWRlIHRoZSBzdHVkZW50IGFuc3dlciB3aXRoIGFuIElOVEVHRVIgc2NvcmUgZnJvbSB7aW50KHNwYWNlWydsbyddKX0gdG8gIgogICAgICAgICAgICAgICAgZiJ7aW50KHNwYWNlWydoaSddKX0gKGhpZ2hlciA9IGJldHRlcikuIFJlcGx5IHdpdGggT05MWSB0aGUgaW50ZWdlci4iKQogICAgZWxzZToKICAgICAgICB0YXNrID0gKGYiR3JhZGUgdGhlIHN0dWRlbnQgYW5zd2VyIHdpdGggYSBzY29yZSBmcm9tIHtzcGFjZVsnbG8nXTouMmZ9IHRvICIKICAgICAgICAgICAgICAgIGYie3NwYWNlWydoaSddOi4yZn0gKGhpZ2hlciA9IGJldHRlcikuIFJlcGx5IHdpdGggT05MWSB0aGUgbnVtYmVyLiIpCiAgICByZXR1cm4gZiJ7Y3R4fVN0dWRlbnQgYW5zd2VyOiB7YW5zfVxuXG57dGFza30iCgoKZGVmIF9wYXJzZSh0ZXh0OiBzdHIsIHNwZWMsIHNwYWNlOiBkaWN0KSAtPiBmbG9hdDoKICAgIHQgPSAodGV4dCBvciAiIikuc3RyaXAoKQogICAgaWYgc3BlYy50YXNrX3R5cGUgPT0gImNsYXNzaWZpY2F0aW9uIjoKICAgICAgICBsb3cgPSB0Lmxvd2VyKCkKICAgICAgICBmb3IgbGFiIGluIHNwYWNlWyJsYWJlbHMiXTogICAgICAgICAgICAgICAgICAgICAgICMgZmlyc3QgbGFiZWwgbWVudGlvbmVkIHdpbnMKICAgICAgICAgICAgaWYgbGFiLmxvd2VyKCkgaW4gbG93OgogICAgICAgICAgICAgICAgcmV0dXJuIGZsb2F0KHNwYWNlWyJ2b2NhYiJdW2xhYl0pCiAgICAgICAgcmV0dXJuIGZsb2F0KHNwYWNlWyJ2b2NhYiJdW3NwYWNlWyJsYWJlbHMiXVswXV0pICAjIGZhbGxiYWNrOiBmaXJzdCBsYWJlbAogICAgbSA9IHJlLnNlYXJjaChyIi0/XGQrKD86XC5cZCspPyIsIHQpCiAgICBtaWQgPSAoc3BhY2VbImxvIl0gKyBzcGFjZVsiaGkiXSkgLyAyLjAKICAgIGlmIG5vdCBtOgogICAgICAgIHJldHVybiByb3VuZChtaWQpIGlmIHNwZWMudGFza190eXBlID09ICJvcmRpbmFsIiBlbHNlIG1pZAogICAgdiA9IG1heChzcGFjZVsibG8iXSwgbWluKHNwYWNlWyJoaSJdLCBmbG9hdChtLmdyb3VwKCkpKSkKICAgIHJldHVybiBmbG9hdChyb3VuZCh2KSkgaWYgc3BlYy50YXNrX3R5cGUgPT0gIm9yZGluYWwiIGVsc2UgdgoKCmRlZiBsb2FkX2xsbSgpOgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2FsTE0sIEF1dG9Ub2tlbml6ZXIKCiAgICB0b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChNT0RFTCkKICAgIGlmIHRvay5wYWRfdG9rZW4gaXMgTm9uZToKICAgICAgICB0b2sucGFkX3Rva2VuID0gdG9rLmVvc190b2tlbgogICAgdG9rLnBhZGRpbmdfc2lkZSA9ICJsZWZ0IiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgY29ycmVjdCBiYXRjaGVkIGRlY29kZXItb25seSBnZW4KICAgIG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIE1PREVMLCB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgIGRldmljZV9tYXA9ImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIG1vZGVsLmV2YWwoKQogICAgcmV0dXJuIHRvaywgbW9kZWwKCgpkZWYgZ2VuZXJhdGUodG9rLCBtb2RlbCwgcHJvbXB0czogbGlzdFtzdHJdKSAtPiBsaXN0W3N0cl06CiAgICBpbXBvcnQgdG9yY2gKCiAgICBzeXNfbXNnID0gIllvdSBhcmUgYSBzdHJpY3QgZ3JhZGluZyBhc3Npc3RhbnQuIEZvbGxvdyB0aGUgcmVxdWVzdGVkIG91dHB1dCBmb3JtYXQgZXhhY3RseS4iCiAgICB0ZXh0cyA9IFt0b2suYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICBbeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50Ijogc3lzX21zZ30sIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBwfV0sCiAgICAgICAgdG9rZW5pemU9RmFsc2UsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlKSBmb3IgcCBpbiBwcm9tcHRzXQogICAgb3V0OiBsaXN0W3N0cl0gPSBbXQogICAgZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKHRleHRzKSwgQkFUQ0gpOgogICAgICAgIGNodW5rID0gdGV4dHNbaTppICsgQkFUQ0hdCiAgICAgICAgZW5jID0gdG9rKGNodW5rLCByZXR1cm5fdGVuc29ycz0icHQiLCBwYWRkaW5nPVRydWUsIHRydW5jYXRpb249VHJ1ZSwKICAgICAgICAgICAgICAgICAgbWF4X2xlbmd0aD0xMDI0KS50byhtb2RlbC5kZXZpY2UpCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIGdlbiA9IG1vZGVsLmdlbmVyYXRlKCoqZW5jLCBtYXhfbmV3X3Rva2Vucz0xMiwgZG9fc2FtcGxlPUZhbHNlLCAgICMgdGVtcCAwIChncmVlZHkpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHBhZF90b2tlbl9pZD10b2sucGFkX3Rva2VuX2lkKQogICAgICAgIGZvciBqIGluIHJhbmdlKGxlbihjaHVuaykpOgogICAgICAgICAgICBvdXQuYXBwZW5kKHRvay5kZWNvZGUoZ2VuW2pdW2VuY1siaW5wdXRfaWRzIl0uc2hhcGVbMV06XSwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKSkKICAgICAgICBwcmludChmIiAgICBnZW5lcmF0ZWQge21pbihpICsgQkFUQ0gsIGxlbih0ZXh0cykpfS97bGVuKHRleHRzKX0iLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIG91dAoKCmRlZiBzY29yZV9kYXRhc2V0KG5hbWU6IHN0ciwgY2ZnLCB0b2ssIG1vZGVsLCBleGFtcGxlczogZGljdCkgLT4gZGljdDoKICAgIHNwZWMgPSBnZXRfc3BlYyhuYW1lKQogICAgcGF0aCA9IGNmZy5wYXRocy5wcm9jZXNzZWQgLyBuYW1lIC8gImVuY29kZXIucGFycXVldCIKICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybiB7InN0YXR1cyI6ICJub19lbmNvZGVyX3BhcnF1ZXQifQogICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQocGF0aCkucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgc3BhY2UgPSBfbGFiZWxfc3BhY2UoZGYsIHNwZWMpCiAgICBkID0gX2V2YWxfcm93cyhkZiwgc3BlYykKICAgIGlmIGQuZW1wdHk6CiAgICAgICAgcmV0dXJuIHsic3RhdHVzIjogIm5vX2V2YWxfcm93cyJ9CgogICAgcHJvbXB0cyA9IFtfcHJvbXB0KHIsIHNwZWMsIHNwYWNlKSBmb3IgXywgciBpbiBkLml0ZXJyb3dzKCldCiAgICByYXcgPSBnZW5lcmF0ZSh0b2ssIG1vZGVsLCBwcm9tcHRzKQogICAgeV9wcmVkID0gbnAuYXJyYXkoW19wYXJzZSh0LCBzcGVjLCBzcGFjZSkgZm9yIHQgaW4gcmF3XSwgZHR5cGU9ZmxvYXQpCiAgICBpZiBzcGVjLnRhc2tfdHlwZSA9PSAiY2xhc3NpZmljYXRpb24iOgogICAgICAgIHlfdHJ1ZSA9IGRbImxhYmVsIl0uYXN0eXBlKHN0cikubWFwKHNwYWNlWyJ2b2NhYiJdKS50b19udW1weShkdHlwZT1mbG9hdCkKICAgIGVsc2U6CiAgICAgICAgeV90cnVlID0gcGQudG9fbnVtZXJpYyhkWyJzY29yZSJdLCBlcnJvcnM9ImNvZXJjZSIpLnRvX251bXB5KGR0eXBlPWZsb2F0KQogICAgZmluID0gbnAuaXNmaW5pdGUoeV90cnVlKSAmIG5wLmlzZmluaXRlKHlfcHJlZCkKICAgIGV4YW1wbGVzW25hbWVdID0gW3sicHJvbXB0IjogcHJvbXB0c1trXSwgImxsbV9yYXciOiByYXdba10sICJwcmVkIjogZmxvYXQoeV9wcmVkW2tdKSwKICAgICAgICAgICAgICAgICAgICAgICAidHJ1ZSI6IGZsb2F0KHlfdHJ1ZVtrXSl9IGZvciBrIGluIHJhbmdlKG1pbigzLCBsZW4ocHJvbXB0cykpKV0KCiAgICBkZWYgbWV0cmljKG1hc2s6IG5wLm5kYXJyYXkpIC0+IGZsb2F0OgogICAgICAgIHJldHVybiBjb21wdXRlX21ldHJpY3MoeV90cnVlW21hc2tdLCB5X3ByZWRbbWFza10sIChzcGVjLmhlYWRsaW5lLCkpLmdldCgKICAgICAgICAgICAgc3BlYy5oZWFkbGluZSwgZmxvYXQoIm5hbiIpKQoKICAgIGlmIHNwZWMucGVyX3Byb21wdDoKICAgICAgICBxaWQgPSBkWyJxdWVzdGlvbl9pZCJdLmFzdHlwZShzdHIpLnRvX251bXB5KCkKICAgICAgICB2YWxzID0gW3YgZm9yIHAgaW4gbnAudW5pcXVlKHFpZFtmaW5dKSBpZiBucC5pc2Zpbml0ZSh2IDo9IG1ldHJpYyhmaW4gJiAocWlkID09IHApKSldCiAgICAgICAgaGVhZGxpbmUgPSBmbG9hdChucC5tZWFuKHZhbHMpKSBpZiB2YWxzIGVsc2UgZmxvYXQoIm5hbiIpCiAgICBlbHNlOgogICAgICAgIGhlYWRsaW5lID0gbWV0cmljKGZpbikKICAgIHJldHVybiB7InN0YXR1cyI6ICJvayIsICJtZXRyaWMiOiBzcGVjLmhlYWRsaW5lLAogICAgICAgICAgICAiaGVhZGxpbmVfc3BsaXQiOiBzcGVjLnRlc3Rfc3BsaXRzWy0xXSBpZiBzcGVjLnByb3RvY29sID09ICJvZmZpY2lhbF9zcGxpdCIgZWxzZSAiY3YiLAogICAgICAgICAgICAibl9ldmFsIjogaW50KGZpbi5zdW0oKSksCiAgICAgICAgICAgICJsbG1faGVhZGxpbmUiOiBOb25lIGlmIG5vdCBucC5pc2Zpbml0ZShoZWFkbGluZSkgZWxzZSByb3VuZChmbG9hdChoZWFkbGluZSksIDQpLAogICAgICAgICAgICAibW9kZWwiOiBNT0RFTH0KCgpkZWYgbWFpbihuYW1lczogbGlzdFtzdHJdKSAtPiBOb25lOgogICAgY2ZnID0gbG9hZF9kYXRhX2NvbmZpZygpCiAgICBuYW1lcyA9IG5hbWVzIG9yIFtuIGZvciBuIGluIFJFR0lTVFJZIGlmIChjZmcucGF0aHMucHJvY2Vzc2VkIC8gbiAvICJlbmNvZGVyLnBhcnF1ZXQiKS5leGlzdHMoKV0KICAgIHRvaywgbW9kZWwgPSBsb2FkX2xsbSgpCiAgICBvdXQsIGV4YW1wbGVzID0ge30sIHt9CiAgICBmb3IgbiBpbiBuYW1lczoKICAgICAgICBwcmludChmIj09PSBMTE0gemVyby1zaG90OiB7bn0gKHtNT0RFTH0pID09PSIsIGZsdXNoPVRydWUpCiAgICAgICAgb3V0W25dID0gc2NvcmVfZGF0YXNldChuLCBjZmcsIHRvaywgbW9kZWwsIGV4YW1wbGVzKQogICAgICAgIHByaW50KGYiICAgIHtufToge291dFtuXX0iLCBmbHVzaD1UcnVlKQogICAgZCA9IGNmZy5wYXRocy5yZXBvcnRzIC8gInBoYXNlX2xsbSIKICAgIGQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKGQgLyAibGxtX2Jhc2VsaW5lLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoCiAgICAgICAgeyJtb2RlbCI6IE1PREVMLCAibWF4X3Jvd3MiOiBNQVhfUk9XUywgInRlbXBlcmF0dXJlIjogMC4wLCAiZGF0YXNldHMiOiBvdXR9LAogICAgICAgIGluZGVudD0yLCBkZWZhdWx0PXN0ciksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAoZCAvICJleGFtcGxlcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKGV4YW1wbGVzLCBpbmRlbnQ9MiwgZGVmYXVsdD1zdHIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcHJpbnQoZiJ3cm90ZSB7ZCAvICdsbG1fYmFzZWxpbmUuanNvbid9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbihbYSBmb3IgYSBpbiBzeXMuYXJndlsxOl0gaWYgbm90IGEuc3RhcnRzd2l0aCgiLSIpXSkK'
)
# 10) LLM zero-shot baseline (Qwen-3B, temp 0). Non-fatal; runs last after the GPU is free.
import base64
dst = os.path.join(REPO, 'experiments', 'llm_zeroshot_baseline.py'); os.makedirs(os.path.dirname(dst), exist_ok=True)
if not os.path.exists(dst):
    open(dst, 'wb').write(base64.b64decode(LLM_BASELINE_B64)); print('wrote LLM baseline script')
env = dict(os.environ, LLM_MODEL='Qwen/Qwen2.5-3B-Instruct', LLM_MAX_ROWS='400', LLM_BATCH='16')
print('LLM exit =', subprocess.run([sys.executable, 'experiments/llm_zeroshot_baseline.py', *DATASETS], cwd=REPO, env=env).returncode)

In [ ]:
# 11) Bundle every new artifact into /kaggle/working (grab it from the Output tab, or it is saved on commit).
import shutil, glob
OUTD = '/kaggle/working/results_bundle'; shutil.rmtree(OUTD, ignore_errors=True); os.makedirs(OUTD)
pats = ['data/processed/*/neural_oof.parquet', 'reports/phase2g/results.*', 'reports/phase2g/preds/*',
        'reports/phase_hybrid/*', 'reports/phase_llm/*', 'reports/phase3/*', 'reports/phase2d/*',
        'reports/phase2f/*', 'reports/figures/*.png']
n = 0
for pat in pats:
    for f in glob.glob(os.path.join(REPO, pat), recursive=True):
        if os.path.isfile(f):
            rel = os.path.relpath(f, REPO); dd = os.path.join(OUTD, rel)
            os.makedirs(os.path.dirname(dd), exist_ok=True); shutil.copy2(f, dd); n += 1
shutil.make_archive('/kaggle/working/results_bundle', 'zip', OUTD)
print(n, 'files ->', '/kaggle/working/results_bundle.zip  (also under /kaggle/working/results_bundle/)')

## After the run - get your results + fill the paper
- Everything is in **`/kaggle/working`**: `results_bundle.zip` + the live `asag/reports/*`. Download from the
  notebook's **Output** tab (or the committed version's Output).
- Unzip over your local repo, run `python paper/verify_numbers.py`, then replace each `\TBD{}` in `paper/main.tex`
  from: `reports/phase_hybrid/three_way.json` (Table 3), `reports/phase2g/results.json` (neural-only),
  `reports/phase3/ablations.json` (-D rows), `reports/phase2f/shap.json`, `reports/phase_llm/llm_baseline.json`.

**No interruption, guaranteed:** with **Persistence = Files only**, if a session ever stops, just open the notebook
and **Run All** again - every finished dataset is skipped and it resumes. Nothing is ever lost.